In [5]:
import json


with open("../config/config.json", "r", encoding="utf-8") as f:
    config = json.load(f)

vocab_size = config["vocab_size"]
max_len = config["max_len"]

In [ ]:
import tensorflow as tf
from keras import layers, Model
from keras.applications import InceptionV3


EMBED_DIM = 512
LSTM_UNITS = 512


# =========================================================
# 1. CNN ENCODER (end-to-end, dentro il modello)
# =========================================================
cnn_base = InceptionV3(include_top=False, pooling='avg')
cnn_base.trainable = False  # oppure True se fine-tuning

img_input = layers.Input(shape=(299, 299, 3), name="image")

img_features = cnn_base(img_input)
img_features = layers.Dense(EMBED_DIM, activation='relu')(img_features)

# =========================================================
# 2. CAPTION INPUT (teacher forcing)
# =========================================================
seq_input = layers.Input(shape=(max_len,), name="caption")   # teacher forcing -> max len - 1

word_emb = layers.Embedding(
    input_dim=vocab_size + 1,  # mask_zero=True -> input_dim=vocabulary.size + 1
    output_dim=EMBED_DIM,
    mask_zero=True
)(seq_input)

# =========================================================
# 3. INITIAL STATE FROM IMAGE
# =========================================================
h0 = layers.Dense(LSTM_UNITS, activation='tanh')(img_features)
c0 = layers.Dense(LSTM_UNITS, activation='tanh')(img_features)

# =========================================================
# 4. LSTM DECODER
# =========================================================
lstm = layers.LSTM(
    LSTM_UNITS,
    return_sequences=True
)

x = lstm(word_emb, initial_state=[h0, c0])

# =========================================================
# 5. OUTPUT VOCAB DISTRIBUTION
# =========================================================
outputs = layers.Dense(vocab_size, activation="softmax")(x)

# =========================================================
# 6. MODEL
# =========================================================
model = Model(inputs=[img_input, seq_input], outputs=outputs)

model.summary()

In [7]:
from models.loss import loss_that_ignores_padding


model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss=loss_that_ignores_padding
)

In [8]:
model.save("../models/model.keras")